# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`
This notebook provides a guideline for loading, exploring, and analyzing the FAIR² dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and utilizes standardized IDs (`@id`) for every entity, field, and column.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.simplefilter('ignore', FutureWarning)

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and their fields. All code references in this notebook use the canonical `@id` identifier for each entity.

**Note:** For Croissant datasets, record sets map to high-level tables within the dataset. Each field (column) also has its own `@id`.

In [ ]:
# List available record sets by their @id.
record_sets = dataset.record_sets
print(f"Record Sets (@id): {', '.join([r['@id'] for r in record_sets])}")

# For each record set, display details
for rs in record_sets:
    recset_id = rs['@id']
    name = rs.get('name', recset_id)
    print(f"---\nRecord Set: {name}\n@id: {recset_id}")
    print("Fields/Columns:")
    for field in rs.get('field', []):
        field_id = field.get('@id', 'N/A')
        field_name = field.get('name', field_id)
        print(f"    - {field_name} (@id: {field_id})")

## 3. Data Extraction
Load data from a specific record set using its `@id` into a pandas DataFrame. You can adapt the code below to load any desired record set, referencing strictly by `@id`.

In [ ]:
# Extract data from all major record sets
dataframes = {}

for rs in dataset.record_sets:
    recset_id = rs['@id']
    print(f"Loading record set: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"Columns: {df.columns.tolist() if not df.empty else 'No records found.'}")
    print()

# Select the primary table for further exploration
# (Choose the largest or most detailed record set @id, or adapt as needed)
if len(dataframes) > 0:
    primary_rs_id = max(dataframes, key=lambda k: dataframes[k].shape[1])  # most columns
    print(f"Primary record set for analysis: {primary_rs_id}")
    df = dataframes[primary_rs_id]
    print(df.head())
else:
    print('No record sets found in dataset.')

## 4. Exploratory Data Analysis (EDA)
Apply common analytic steps: filter for numeric fields, remove outliers, normalize data, and group/categorize. All field access should use the correct `@id` from the earlier overview.

In [ ]:
# Pick a candidate numeric field and a grouping field (fill in the @id as identified above)
# This example attempts common medical/demographic fields. Adjust as per actual field @ids from overview.

# Print all available column @ids for selection
print(f"Columns in DataFrame ({primary_rs_id}):")
print(df.columns.tolist())

# Select one likely numeric field (@id) and one grouping field (@id) if present
numeric_field_id = None
group_field_id = None
# Attempt to infer from names
for col in df.columns:
    lower = col.lower()
    if (numeric_field_id is None and (('age' in lower or 'interval' in lower or 'distance' in lower or 'score' in lower) and not df[col].isnull().all())):
        numeric_field_id = col
    if (group_field_id is None and ('sex' in lower or 'msi' in lower or 'status' in lower or 'group' in lower)):
        group_field_id = col

print(f"Selected numeric field (@id): {numeric_field_id}")
print(f"Selected group/categorical field (@id): {group_field_id}")

if numeric_field_id and numeric_field_id in df.columns:
    # Drop rows with missing values in the numeric field
    filtered_df = df[df[numeric_field_id].notnull()].copy()

    # Attempt to convert to numeric
    filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    filtered_df = filtered_df[filtered_df[numeric_field_id].notnull()]

    # Remove outliers using a Z-score approach (e.g., >3 std from mean)
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df['zscore'] = (filtered_df[numeric_field_id] - mean) / std
    filtered_df = filtered_df[filtered_df['zscore'].abs() < 3]
    print(f"Filtered (outliers removed) {numeric_field_id} stats: min={filtered_df[numeric_field_id].min()}, max={filtered_df[numeric_field_id].max()}, std={filtered_df[numeric_field_id].std():.2f}")

    # Normalize field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by categorical/group field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count', 'std'])
        print(f"\nGrouped by {group_field_id}:")
        print(grouped_df)
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize distributions and relationships for selected fields.

- Histogram of numeric distribution
- Boxplot by group
- Barplot of group counts, if applicable

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    # Histogram
    sns.histplot(filtered_df[numeric_field_id], kde=True, ax=axs[0], color='tab:blue')
    axs[0].set_title(f"Distribution of {numeric_field_id}")
    # Boxplot by group if available
    if group_field_id and group_field_id in filtered_df.columns:
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id], ax=axs[1])
        axs[1].set_title(f"{numeric_field_id} by {group_field_id}")
    else:
        sns.boxplot(y=filtered_df[numeric_field_id], ax=axs[1])
        axs[1].set_title(f"Boxplot of {numeric_field_id}")
    plt.tight_layout()
    plt.show()

    # Barplot group counts
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(6,3))
        sns.countplot(y=filtered_df[group_field_id], order=filtered_df[group_field_id].value_counts().index)
        plt.title(f"Record count by {group_field_id}")
        plt.show()

## 6. Conclusion
We have demonstrated how to:
- Load a Croissant dataset using its schema URL with `mlcroissant`.
- Discover and reference every entity (record set, field, column) by its canonical `@id`.
- Extract and analyze data in a pandas DataFrame, filtering and grouping using these `@id`s.
- Visualize the structure and summary statistics.

**Next steps:** Explore relationships between multiple record sets (by joining via `@id`s if needed), or apply domain-specific analyses using the metadata provided. Always check the original dataset documentation for precise column and field meanings.